# ENG-03: Artifact Rejection (ICA)

This notebook walks through every step of the ENG-03 pipeline on **one patient session**,
with plots at each stage so you can visually verify the results.

**Prerequisite:** ENG-02 aligned events must exist for the patient you choose.

In [ ]:
import os
import sys

# This notebook lives in awaken-ai/eda/.
# Go up one level to awaken-ai/ so that "from src.…" imports work.
os.chdir(os.path.join(os.path.abspath("."), ".."))
sys.path.insert(0, os.getcwd())

import matplotlib.pyplot as plt  # noqa: E402
import mne  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

%matplotlib inline
mne.set_log_level("WARNING")
print(f"MNE {mne.__version__}  •  NumPy {np.__version__}  •  pandas {pd.__version__}")

## 0. Configure — pick a patient + session

Change `PATIENT_ID` and `SESSION_DATE` below to match a patient for which ENG-02 has produced aligned events.

In [ ]:
PATIENT_ID = "CON008"  # <── change me
SESSION_DATE = "2025-08-14"  # <── change me

## 1. Load inputs  (`UnifiedDataLoader` + aligned events)

In [ ]:
from src.data_loading import config
from src.data_loading.unified_data_loader import UnifiedDataLoader

loader = UnifiedDataLoader(verbose=False)

# Load EDF
raw = loader.load_edf(PATIENT_ID, date=SESSION_DATE, use_clipped=True)
print(f"EDF loaded: {len(raw.ch_names)} channels, {raw.times[-1]:.1f}s, sfreq={raw.info['sfreq']}Hz")
print(f"Channels: {raw.ch_names}")

In [ ]:
# Load aligned events from ENG-02
aligned_path = config.ALIGNED_EVENTS_DIR / f"{PATIENT_ID}_events.parquet"
# .resolve() follows the symlink so pandas can open the real file
aligned_df = pd.read_parquet(aligned_path.resolve())
session_df = aligned_df[aligned_df["date"] == SESSION_DATE].copy()

print(f"Aligned events: {len(session_df)} trials for {PATIENT_ID} on {SESSION_DATE}")
session_df.groupby("trial_type").size()

## 2. Inspect channels — which are EEG vs non-EEG?

In [ ]:
from src.data_processing.artifact_rejection import _find_eog_channels, _pick_eeg_indices
from src.utils.signal_processing import exclude_non_eeg_channels

non_eeg = exclude_non_eeg_channels(raw)
eeg_picks = _pick_eeg_indices(raw)
eeg_names = [raw.ch_names[i] for i in eeg_picks]
eog_chs = _find_eog_channels(raw)

print(f"Total channels in EDF: {len(raw.ch_names)}")
print(f"Non-EEG channels EXCLUDED ({len(non_eeg)}): {non_eeg}")
print(f"EEG channels for ICA ({len(eeg_names)}): {eeg_names}")
print(f"EOG reference channels (for blink detection): {eog_chs}")

### Plot: Raw EEG (first 10 seconds, before any cleaning)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
start_sec, end_sec = 0, 10
data_raw = raw.copy().pick(eeg_names).get_data(tmin=start_sec, tmax=end_sec) * 1e6  # to µV
times = np.linspace(start_sec, end_sec, data_raw.shape[1])

offsets = np.arange(len(eeg_names)) * 150  # µV spacing
for i, ch in enumerate(eeg_names):
    ax.plot(times, data_raw[i] + offsets[i], lw=0.5, label=ch)

ax.set_xlabel("Time (s)")
ax.set_ylabel("Channel (µV, offset for visibility)")
ax.set_title(f"{PATIENT_ID} — Raw EEG (first {end_sec}s, BEFORE ICA)")
ax.set_yticks(offsets)
ax.set_yticklabels(eeg_names, fontsize=8)
plt.tight_layout()
plt.show()

## 3. Timezone offset detection

In [ ]:
from src.utils.time_utils import detect_timezone_offset, unix_to_edf

tz_offset = detect_timezone_offset(raw, session_df)
edf_start_unix = raw.info["meas_date"].timestamp()

print(f"EDF meas_date (unix): {edf_start_unix:.2f}")
print(f"First trial start (unix): {session_df['start_time'].min():.2f}")
print(f"Detected timezone offset: {tz_offset:.0f}s  ({tz_offset / 3600:.1f}h)")

# Quick sanity check: convert first trial start → EDF seconds
first_start_edf = unix_to_edf(session_df["start_time"].min(), edf_start_unix=edf_start_unix, timezone_offset=tz_offset)
print(f"First trial in EDF time: {first_start_edf:.2f}s")

## 4. Fit & apply ICA (session-level)

In [ ]:
from src.data_processing.artifact_rejection import ArtifactRejector

# We instantiate just to borrow _apply_ica; we won't call run_session yet.
ar = ArtifactRejector(verbose=False)

raw_clean, ica_summary = ar._apply_ica(raw)

print("─── ICA Summary ───")
print(f"ICA method:          {ica_summary.method}")
print(f"Classification:      {ica_summary.classification_method}")
print(f"Components fit:      {ica_summary.n_components}")
print(f"EOG channels used:   {ica_summary.eog_channels_used}")
print(f"EOG components:      {ica_summary.eog_components}")
print(f"ECG channels used:   {ica_summary.ecg_channels_used}")
print(f"ECG components:      {ica_summary.ecg_components}")
print(f"Muscle components:   {ica_summary.muscle_components}")
print(f"Line noise comps:    {ica_summary.line_noise_components}")
print(f"Channel noise comps: {ica_summary.channel_noise_components}")
print(f"TOTAL excluded:      {ica_summary.excluded}")

if ica_summary.iclabel_labels:
    print("\n─── ICLabel per-component labels ───")
    for idx, label in enumerate(ica_summary.iclabel_labels):
        marker = " ** EXCLUDED **" if idx in ica_summary.excluded else ""
        print(f"  IC{idx:02d}: {label}{marker}")

print("\nNotes:")
for n in ica_summary.notes:
    print(f"  {n}")

# Quick sanity: how much signal changed?
seg_before = raw.copy().pick(eeg_names).get_data(tmin=0, tmax=30) * 1e6
seg_after = raw_clean.copy().pick(eeg_names).get_data(tmin=0, tmax=30) * 1e6
mad = np.mean(np.abs(seg_before - seg_after))
print(f"\nMean absolute difference (first 30s, uV): {mad:.3f}")
if mad < 0.01:
    print("WARNING: Difference ~ 0 -- ICA may not have excluded any components.")

### Plot: Before vs After ICA (overlay same 10 seconds)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

data_before = raw.copy().pick(eeg_names).get_data(tmin=start_sec, tmax=end_sec) * 1e6
data_after = raw_clean.copy().pick(eeg_names).get_data(tmin=start_sec, tmax=end_sec) * 1e6
times = np.linspace(start_sec, end_sec, data_before.shape[1])

n_excluded = len(ica_summary.excluded)
for ax, data, title in [
    (axes[0], data_before, "BEFORE ICA"),
    (axes[1], data_after, f"AFTER ICA  ({n_excluded} component(s) removed)"),
]:
    offsets = np.arange(len(eeg_names)) * 150
    for i, ch in enumerate(eeg_names):
        ax.plot(times, data[i] + offsets[i], lw=0.5)
    ax.set_title(f"{PATIENT_ID} — {title}  (first {end_sec}s)")
    ax.set_yticks(offsets)
    ax.set_yticklabels(eeg_names, fontsize=8)
    ax.set_ylabel("µV (offset)")

axes[1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

# Per-channel difference summary
diff_per_ch = np.mean(np.abs(data_before - data_after), axis=1)
print("Per-channel mean |diff| (µV):")
for ch, d in zip(eeg_names, diff_per_ch):
    print(f"  {ch:>5s}: {d:.2f}")

### Plot: Difference signal (what ICA removed)

In [ ]:
diff = data_before - data_after

fig, ax = plt.subplots(figsize=(14, 6))
offsets = np.arange(len(eeg_names)) * 50
for i, ch in enumerate(eeg_names):
    ax.plot(times, diff[i] + offsets[i], lw=0.5, label=ch)

ax.set_title(f"{PATIENT_ID} — Signal REMOVED by ICA (first {end_sec}s)")
ax.set_xlabel("Time (s)")
ax.set_ylabel("µV (offset)")
ax.set_yticks(offsets)
ax.set_yticklabels(eeg_names, fontsize=8)
plt.tight_layout()
plt.show()

## 5. Build fixed-window epochs for one trial type

In [ ]:
from src.data_processing.artifact_rejection import _trial_type_window_sec

# Pick the trial type with the most trials for a meaningful demo
tt_counts = session_df.groupby(session_df["trial_type"].str.lower()).size().sort_values(ascending=False)
print("Trial type counts:")
print(tt_counts)

DEMO_TT = tt_counts.index[0]
print(f'\nUsing trial type: "{DEMO_TT}"  (window = {_trial_type_window_sec(DEMO_TT, None)}s)')

In [ ]:
tt_df = session_df[session_df["trial_type"].str.lower() == DEMO_TT].copy()

epochs = ar._build_fixed_window_epochs(
    raw_clean,
    tt_df,
    trial_type=DEMO_TT,
    edf_start_unix=edf_start_unix,
    timezone_offset=tz_offset,
)

print(f"Epochs created: {len(epochs)}")
print(f"Channels in epochs: {epochs.ch_names}")
print(f"Epoch time window: {epochs.tmin}s → {epochs.tmax}s")
print(f"Data shape: {epochs.get_data().shape}  (n_epochs, n_channels, n_times)")

### Plot: A few individual epochs (overlay channels)

In [ ]:
n_show = min(4, len(epochs))
fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

ep_data = epochs.get_data() * 1e6  # µV
ep_times = epochs.times

for i in range(n_show):
    ax = axes[i]
    for ch_idx in range(ep_data.shape[1]):
        ax.plot(ep_times, ep_data[i, ch_idx], lw=0.4, alpha=0.7)
    ax.set_title(f"Epoch {i} ({DEMO_TT})", fontsize=10)
    ax.set_ylabel("µV")

axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"{PATIENT_ID} — Individual Epochs (EEG-only, ICA-cleaned)", y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## 6. Peak-to-peak (PTP) distribution & auto-rejection

In [ ]:
from src.data_processing.artifact_rejection import _epoch_ptp_uv

ptp_uv = _epoch_ptp_uv(epochs)
threshold = np.percentile(ptp_uv, 95)

print("PTP distribution (µV):")
print(f"  min={ptp_uv.min():.1f}  median={np.median(ptp_uv):.1f}  p95={threshold:.1f}  max={ptp_uv.max():.1f}")
print(f"Epochs above p95 threshold: {(ptp_uv > threshold).sum()} / {len(ptp_uv)}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["red" if v > threshold else "steelblue" for v in ptp_uv]
ax.bar(range(len(ptp_uv)), ptp_uv, color=colors, width=1.0, edgecolor="none")
ax.axhline(threshold, color="orange", ls="--", lw=2, label=f"P95 threshold = {threshold:.0f} µV")
ax.set_xlabel("Epoch index")
ax.set_ylabel("Max peak-to-peak (µV)")
ax.set_title(f"{PATIENT_ID} — PTP per epoch ({DEMO_TT})  •  red = will be DROPPED")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Now actually apply the rejection
epochs_clean, thresh_uv, dropped_idx = ar._auto_reject_epochs(epochs.copy())

print(f"Before: {len(epochs)} epochs")
print(f"Threshold: {thresh_uv:.1f} µV (p95)")
print(f"Dropped: {len(dropped_idx)} epochs (indices: {dropped_idx})")
print(f"After: {len(epochs_clean)} epochs")

## 7. Grand average (across surviving epochs)

This is the simplest sanity check: average all kept epochs to see if a clean signal emerges.

In [ ]:
evoked = epochs_clean.average()

fig, ax = plt.subplots(figsize=(12, 5))
evoked_data = evoked.data * 1e6  # µV
for i, ch in enumerate(evoked.ch_names):
    ax.plot(evoked.times, evoked_data[i], lw=0.8, label=ch)

ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude (µV)")
ax.set_title(f"{PATIENT_ID} — Grand Average ({DEMO_TT}, {len(epochs_clean)} epochs)")
ax.legend(fontsize=7, ncol=4, loc="upper right")
plt.tight_layout()
plt.show()

## 8. QC metadata preview

This is the same data that gets saved to `eng03_qc.parquet` — one row per trial type.

In [ ]:
ptp_stats = ar._ptp_stats(ptp_uv)

qc = ar._qc_row(
    patient_id=PATIENT_ID,
    date=SESSION_DATE,
    trial_type=DEMO_TT,
    n_total=len(epochs),
    n_dropped=len(dropped_idx),
    threshold_uv=thresh_uv,
    ica_summary=ica_summary,
    notes=[],
    ptp_stats=ptp_stats,
    drop_reason="ENG03_PTP_GT_P95" if dropped_idx else None,
)

qc_df = pd.DataFrame([qc])
print("QC row columns:", list(qc_df.columns))
qc_df.T

## 9. (Optional) Run the full pipeline end-to-end and inspect outputs

This saves `.fif` epochs + QC parquet to disk.

In [ ]:
# Uncomment the lines below to run the full pipeline and save to disk:

# ar_full = ArtifactRejector(verbose=False)
# saved = ar_full.run_session(PATIENT_ID, SESSION_DATE, save=True)
# print('Saved epoch files:')
# for tt, path in saved.items():
#     print(f'  {tt}: {path}')
#
# # Read back the QC parquet
# qc_path = ar_full._qc_output_path(PATIENT_ID, SESSION_DATE)
# if qc_path.exists():
#     qc_full = pd.read_parquet(qc_path)
#     display(qc_full)